# Hybrid current/voltage charging: four evolving Pareto fronts
DFN + partially reversible plating (OKane2022), stored SOC 20→80%. Each generation displays observed feasible nondominated fronts for cumulative local net deposition, peak metallic lithium, irreversible dead lithium and time-integrated metallic lithium inventory. Primary selection uses time and gross deposition; the three other fronts are projections of the evaluated archive, not independently optimized fronts. Set OBJECTIVE to any risk column for a dedicated search. No simulated results are bundled. Model risk proxies are not failure probabilities.

In [ ]:
import sys, subprocess, os
print('Hybrid Pareto v1 | Python:', sys.version)
if any(k in sys.modules for k in ('jax','jaxlib','pybamm')):
    raise RuntimeError('Use Runtime > Disconnect and delete runtime, then reconnect and run all.')
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax', 'jaxlib'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pybamm==25.12.2'])
import importlib.util
if importlib.util.find_spec('jax') is not None:
    raise RuntimeError('JAX remains installed; use a clean dedicated runtime.')
os.environ['PYBAMM_DISABLE_TELEMETRY'] = 'true'
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
import pybamm, time, json, pathlib
from scipy.stats import norm
from scipy.integrate import trapezoid
from IPython.display import display
assert pybamm.__version__ == '25.12.2', pybamm.__version__
pybamm.set_logging_level('ERROR')
torch.set_num_threads(1)
print('Hybrid Pareto v1 ready:', pybamm.__version__, torch.__version__)


## Configuration and calibration
Start with one run; change N_RUNS to 20 for ensembles. All methods receive the same initial design per run and representation. Each generation evaluates OFFSPRING protocols, including failed simulations.

In [ ]:
N_RUNS=1
N_GEN=5
N_INIT=12
OFFSPRING=4
POOL=48
HORIZON=10800.
RAMP=1.
V_MIN,V_MAX=3.4,4.2
I_MAX=15.
SOC0,SOC_TARGET=.2,.8
TEMP=298.15
MODEL='DFN'
BASE_SEED=42
MIN_DT=60.
MAX_SEG=8
N_WAVE=32
HARM=3
ENC_N=128
METHODS=['Random','GA','DKL-GA']
FAMILIES=['direct','fourier','constant']
OBJECTIVE='gross_mAh'
# Fixed hypervolume box: preliminary scales, freeze before comparison.
REF=np.array([180.,100.]) # minutes, mAh (for gross/peak/dead)
EXPOSURE_REF=18000. # mAh min
OUT=pathlib.Path('hybrid_pareto_results'); OUT.mkdir(exist_ok=True)
CONC = 'Negative electrode volume-averaged concentration [mol.m-3]'
PLATE = 'Loss of capacity to negative lithium plating [A.h]'

def make_model(kind=MODEL):
    return getattr(pybamm.lithium_ion, kind)(options={'lithium plating':'partially reversible'})
def params():
    p = pybamm.ParameterValues('OKane2022')
    p.update({'Ambient temperature [K]':TEMP,'Initial temperature [K]':TEMP,
              'Upper voltage cut-off [V]':4.21})
    return p

def solver(): return pybamm.IDAKLUSolver(rtol=1e-6, atol=1e-8)
def series(sol, name):
    a = np.asarray(sol[name].entries).squeeze()
    if a.ndim != 1 or len(a) != len(sol.t): raise ValueError(f'Unexpected shape {name}: {a.shape}')
    return a

def calibrate(soc):
    m = make_model(); p=params()
    exp=pybamm.Experiment([pybamm.step.current(0.,duration=1.,period=1.)])
    sol=pybamm.Simulation(m,parameter_values=p,experiment=exp,solver=solver()).solve(initial_soc=soc)
    return float(series(sol,CONC)[0]),float(series(sol,'Voltage [V]')[0])
C0, OCV0 = calibrate(SOC0)
CT, OCVT = calibrate(SOC_TARGET)
assert CT > C0
print('Calibration:', C0, CT, OCV0, OCVT)
from copy import deepcopy
from scipy.integrate import cumulative_trapezoid
from IPython.display import clear_output
RISK={'gross_mAh':'Cumulative net deposition (mAh)','peak_mAh':'Peak metallic lithium (mAh)','dead_mAh':'Dead lithium (mAh)','exposure_mAh_min':'Integrated plating exposure (mAh min)'}
assert OBJECTIVE in RISK
assert N_INIT>=8 and N_RUNS>=1


## Genotype and operations
Each segment has a mode, duration and normalized waveform. Current command is -I_MAX*u (charging-only); voltage is V_MIN+(V_MAX-V_MIN)*u. Discharge under voltage control is allowed within absolute current bounds and explicitly recorded. Constant family uses four piecewise holds per segment; direct uses 32 samples with linear interpolation; Fourier uses DC + three sine/cosine pairs with bounded decoded output. Structural mutations operate on segment lists, including split, delete, duplicate, reorder, mode flip and breakpoint moves. Protocol crossover exchanges segment blocks; same-mode waveform crossover uses one/two-point or smooth mixing. Fourier waveform crossover is in coefficient space. Minimum duration and total horizon are repaired; no smoothness penalty.

In [ ]:
def grid(n=N_WAVE): return np.linspace(0,1,n)
def basis(t):
    return np.column_stack([np.ones(len(t))]+[f(2*np.pi*k*t) for k in range(1,HARM+1) for f in (np.cos,np.sin)])
def decode(s,t):
    a=np.asarray(s['a']); t=np.asarray(t)
    if s['family']=='fourier': y=basis(t)@a
    elif s['family']=='constant': y=a[np.minimum((t*len(a)).astype(int),len(a)-1)]
    else: y=np.interp(t,grid(len(a)),a)
    return np.clip(y,0,1)
def fit_wave(y,family):
    y=np.asarray(y)
    if family=='fourier': return np.linalg.lstsq(basis(grid(len(y))),y,rcond=None)[0].tolist()
    if family=='constant': return [float(z.mean()) for z in np.array_split(y,4)]
    return np.interp(grid(),grid(len(y)),y).tolist()
def seg(mode,d,level,family,rng=None):
    y=np.full(N_WAVE,level)
    if rng is not None: y+=np.interp(grid(),grid(5),rng.normal(0,.07,5))
    return dict(mode=mode,d=float(d),family=family,a=fit_wave(np.clip(y,0,1),family))
def repair(p):
    p=deepcopy(p[:MAX_SEG]); d=np.array([max(0,s['d']-MIN_DT) for s in p])
    d=np.ones(len(p)) if d.sum()==0 else d
    d=MIN_DT+(HORIZON-len(p)*MIN_DT)*d/d.sum()
    for s,v in zip(p,d): s['d']=float(v)
    return p
def random_protocol(rng,f):
    k=int(rng.integers(1,MAX_SEG+1)); modes=rng.choice(['I','V'],k)
    return repair([seg(m,HORIZON*w,rng.uniform(.08,.6) if m=='I' else rng.uniform(.65,1),f,rng) for m,w in zip(modes,rng.dirichlet(np.ones(k)))])
def initial(rng,f):
    v=lambda x:(x-V_MIN)/(V_MAX-V_MIN)
    p=[[seg('V',HORIZON,v(x),f)] for x in [3.9,4.,4.1,4.2]]
    p += [[seg('I',d,i/I_MAX,f),seg('V',HORIZON-d,1.,f)] for d,i in [(1800,2.5),(1200,5),(900,7.5)]]
    p += [[seg('I',900,2.5/I_MAX,f),seg('V',HORIZON-900,v(4.1),f)]]
    return [repair(x) for x in p]+[random_protocol(rng,f) for _ in range(N_INIT-len(p))]
def mutate(p,rng):
    p=deepcopy(p); k=int(rng.integers(len(p))); s=p[k]; f=s['family']
    op=rng.choice(['gaussian','step','replace','scale','coefficient','breakpoint','flip','split','delete','duplicate','reorder'])
    if op=='breakpoint' and len(p)>1:
        j=min(k,len(p)-2); delta=rng.normal(0,.2*min(p[j]['d'],p[j+1]['d']))
        delta=np.clip(delta,MIN_DT-p[j]['d'],p[j+1]['d']-MIN_DT); p[j]['d']+=delta;p[j+1]['d']-=delta
    elif op=='flip':
        p[k]=seg('V' if s['mode']=='I' else 'I',s['d'],rng.uniform(.7,1) if s['mode']=='I' else rng.uniform(.05,.5),f,rng)
    elif op=='split' and len(p)<MAX_SEG:
        a,b=deepcopy(s),deepcopy(s); q=rng.uniform(.25,.75);a['d']*=q;b['d']*=1-q
        a['a']=fit_wave(decode(s,grid()*q),f);b['a']=fit_wave(decode(s,q+(1-q)*grid()),f);p[k:k+1]=[a,b]
    elif op=='delete' and len(p)>1: p.pop(k)
    elif op=='duplicate' and len(p)<MAX_SEG: p.insert(k,deepcopy(s))
    elif op=='reorder' and len(p)>1:
        j=int(rng.integers(len(p)));p[k],p[j]=p[j],p[k]
    elif op=='coefficient' and f=='fourier':
        j=int(rng.integers(len(s['a'])));s['a'][j]+=rng.normal(0,.08)
    else:
        x=grid();y=decode(s,x);i,j=sorted(rng.choice(N_WAVE+1,2,replace=False))
        if op=='gaussian': y+=rng.normal(0,.15)*np.exp(-.5*((x-rng.random())/rng.uniform(.03,.2))**2)
        elif op=='replace': y[i:j]=rng.uniform(0,1)
        elif op=='scale': y[i:j]*=rng.uniform(.6,1.4)
        else: y[i:j]+=rng.normal(0,.12)
        s['a']=fit_wave(np.clip(y,0,1),f)
    return repair(p),str(op)
def cross(a,b,rng):
    a,b=deepcopy(a),deepcopy(b)
    op=rng.choice(['protocol_one','protocol_two','wave_one','wave_two','wave_smooth'])
    if op=='protocol_one':
        i=int(rng.integers(1,len(a)+1));j=int(rng.integers(len(b)));p=a[:i]+b[j:]
    elif op=='protocol_two':
        i,j=sorted(rng.choice(len(a)+1,2,replace=False));k,l=sorted(rng.choice(len(b)+1,2,replace=False));p=a[:i]+b[k:l]+a[j:]
    else:
        p=a; matches=[(i,j) for i,s in enumerate(a) for j,t in enumerate(b) if s['mode']==t['mode'] and s['family']==t['family']]
        if matches:
            i,j=matches[int(rng.integers(len(matches)))];s,t=a[i],b[j];f=s['family']
            x,y=np.array(s['a']),np.array(t['a'])
            k,l=sorted(rng.choice(np.arange(1,len(x)),2,replace=False))
            if op=='wave_one':x[k:]=y[k:]
            elif op=='wave_two':x[k:l]=y[k:l]
            else:
                w=1/(1+np.exp(-(np.arange(len(x))-k)/1.5));x=(1-w)*x+w*y
            s['a']=x.tolist()
    return repair(p),str(op)
def encode(p):
    t=(np.arange(ENC_N)+.5)*HORIZON/ENC_N; z=np.zeros((4,ENC_N));start=0.
    for s in p:
        end=start+s['d']; mask=(t>=start)&(t<end); i=0 if s['mode']=='I' else 1
        z[i,mask]=1;z[i+2,mask]=decode(s,(t[mask]-start)/s['d']);start=end
    return z.ravel()
rng=np.random.default_rng(2)
for f in FAMILIES:
    p=random_protocol(rng,f)
    for _ in range(100):
        p,_=mutate(p,rng);p,_=cross(p,random_protocol(rng,f),rng)
        assert abs(sum(s['d'] for s in p)-HORIZON)<1e-6 and min(s['d'] for s in p)>=MIN_DT-1e-7
        assert np.isfinite(encode(p)).all()


## Electrochemical evaluation
Each segment uses a separate one-step experiment with the preceding solution passed as starting_solution, preserving internal states. Stop immediately at the stored-charge event. Reject early stops due to other events, sampled voltage/current violations and incomplete charging. Spatial integration uses the DFN finite-volume cell widths; local negative net plating current is integrated before spatial averaging. Metallic and dead lithium concentrations are converted to mAh using electrode area and Faraday constant. The capacity-loss variable is NOT substituted for either separate inventory. A mass-balance diagnostic checks net plating current against metallic + dead lithium changes. Smoke tests verify actual model variables before the campaign.

In [ ]:
m=make_model(); p=params()
def resolve(*names):
    for name in names:
        if name in m.variables: return name
    raise KeyError(f'Missing {names}; available: '+str([k for k in m.variables if 'plating' in k.lower() or 'dead lithium' in k.lower()]))
MET=resolve('Negative electrode lithium plating concentration [mol.m-3]','Lithium plating concentration [mol.m-3]')
DEAD=resolve('Negative electrode dead lithium concentration [mol.m-3]','Dead lithium concentration [mol.m-3]')
FLUX=resolve('Negative electrode lithium plating volumetric interfacial current density [A.m-3]','Lithium plating volumetric interfacial current density [A.m-3]')
AREA=float(p.evaluate(m.param.L_y*m.param.L_z)); F=float(p.evaluate(m.param.F))
print('Resolved:',MET,DEAD,FLUX, 'Area:',AREA)
def drive_segment(s):
    d=s['d']
    if s['family']=='constant':
        a=np.asarray(s['a']);t=[0.];y=[a[0]]
        for k in range(1,len(a)):
            edge=d*k/len(a);t.extend([edge-min(RAMP,d/len(a)/2),edge]);y.extend([a[k-1],a[k]])
        t.append(d);y.append(a[-1]);t=np.asarray(t);y=np.clip(y,0,1)
    else:
        t=np.linspace(0,d, max(N_WAVE, 16*HARM+1));y=decode(s,t/d)
    u=-I_MAX*y if s['mode']=='I' else V_MIN+(V_MAX-V_MIN)*y
    return np.column_stack([t,u])
def termination():
    return pybamm.step.CustomTermination('Stored charge target',lambda v:CT-v[CONC])
def spatial(sol,name,dx):
    a=np.asarray(sol[name].entries)
    if a.shape!=(len(dx),len(sol.t)): raise ValueError(f'{name} shape {a.shape}, expected {(len(dx),len(sol.t))}')
    return a
def evaluate(protocol,trace=False):
    start=time.perf_counter(); chunks=[]; previous=None; wall=0.; reached=False;error=''
    try:
        for s in protocol:
            constructor=pybamm.step.current if s['mode']=='I' else pybamm.step.voltage
            st=constructor(drive_segment(s),duration=s['d'],period=2.,termination=termination(),skip_ok=False)
            sim=pybamm.Simulation(make_model(),parameter_values=params(),solver=solver(),experiment=pybamm.Experiment([st]))
            kwargs={'initial_soc':SOC0} if previous is None else {'starting_solution':previous}
            previous=sim.solve(calc_esoh=False,**kwargs)
            sol=previous.cycles[-1].steps[-1]
            t=np.asarray(sol.t); C=series(sol,CONC); I=series(sol,'Current [A]');V=series(sol,'Voltage [V]')
            dx=np.diff(sim.mesh['negative electrode'].edges)
            metal=spatial(sol,MET,dx); dead=spatial(sol,DEAD,dx); j=spatial(sol,FLUX,dx)
            Q=AREA*F/3.6*(dx@metal); D=AREA*F/3.6*(dx@dead)
            G=AREA/3.6*(dx@np.maximum(-j,0)); net=AREA/3.6*(dx@(-j))
            if not np.isfinite(np.stack([t,C,I,V,Q,D,G,net])).all(): raise ValueError('Nonfinite trace')
            hit=np.flatnonzero(C>=CT-1e-5*(CT-C0)); k=int(hit[0])+1 if len(hit) else len(t)
            chunks.append(dict(t=t[:k],C=C[:k],I=I[:k],V=V[:k],Q=Q[:k],D=D[:k],G=G[:k],net=net[:k],mode=np.full(k,s['mode']),local=np.max(metal[:,:k],axis=0)))
            reached=bool(len(hit))
            if reached: break
            if t[-1]-t[0]<s['d']-0.05: raise ValueError('Unscheduled early step termination: '+str(sol.termination))
        a={key:np.concatenate([c[key] for c in chunks]) for key in chunks[0]}
        t,C,I,V,Q,D=[a[key] for key in ('t','C','I','V','Q','D')]
        # Integrate separately per segment, avoiding quadrature across command discontinuities.
        gross=sum(trapezoid(c['G'],c['t']) for c in chunks)
        net=sum(trapezoid(c['net'],c['t']) for c in chunks)
        exposure=sum(trapezoid(np.maximum(c['Q'],0),c['t'])/60 for c in chunks)
        residual=float(net-(Q[-1]+D[-1]-Q[0]-D[0]))
        progress=float((C[-1]-C0)/(CT-C0)); imax=float(np.max(np.abs(I)))
        violation=max(0,1-progress)+max(0,imax/I_MAX-1)+max(0,(np.max(V)-V_MAX)/.1)+max(0,(V_MIN-np.min(V))/.1)
        feasible=bool(reached and imax<=I_MAX+1e-3 and np.max(V)<=V_MAX+1e-5 and np.min(V)>=V_MIN-1e-5)
        row=dict(feasible=feasible,violation=violation,time_min=float(t[-1]/60),gross_mAh=float(gross),peak_mAh=float(max(0,Q.max())),dead_mAh=float(max(0,D[-1]-D[0])),exposure_mAh_min=float(exposure),final_mAh=float(Q[-1]),balance_residual_mAh=residual,max_A=imax,progress=progress,local_peak=float(a['local'].max()),voltage_fraction=sum(c['t'][-1]-c['t'][0] for c in chunks if c['mode'][0]=='V')/max(t[-1],1e-9),segments_used=len(chunks),error='',wall_s=time.perf_counter()-start)
        if trace: row['trace']=a
        return row
    except Exception as e:
        return dict(feasible=False,violation=100.,time_min=np.nan,**{k:np.nan for k in RISK},error=f'{type(e).__name__}: {e}',wall_s=time.perf_counter()-start)
smoke=initial(np.random.default_rng(1),'direct')
for idx in [2,4]:
    r=evaluate(smoke[idx],trace=True);print({k:v for k,v in r.items() if k!='trace'})
    if r['error']: raise RuntimeError(r['error'])


## Pareto sorting, hypervolume and DKL
Feasible nondominated points only. GA tournaments use nondominated rank and crowding; failed/infeasible protocols compete by violation when no feasible points exist. DKL fits independent exact deep-kernel GPs for time and selected risk using feasible data, plus a separate GP regression feasibility score on all observations (heuristic, not calibrated probability). Monte Carlo constrained EHVI ranks a genetic pool. If insufficient feasible data exist, use GA. Every evaluation counts toward budget.

In [ ]:
class DKL(nn.Module):
    def __init__(self):
        super().__init__(); self.encoder=nn.Sequential(nn.Linear(4*ENC_N,32),nn.Tanh(),nn.Linear(32,4),nn.Tanh()).double()
        self.log_length=nn.Parameter(torch.tensor(-.2,dtype=torch.double)); self.log_amp=nn.Parameter(torch.tensor(0.,dtype=torch.double)); self.log_noise=nn.Parameter(torch.tensor(-4.,dtype=torch.double))
    def kernel(self,a,b):
        za,zb=self.encoder(a),self.encoder(b)
        d=torch.cdist(za,zb).square(); return self.log_amp.exp()*torch.exp(-.5*d/self.log_length.exp().square())
    def factor(self,x):
        K=self.kernel(x,x)+(self.log_noise.exp()+1e-6)*torch.eye(len(x),dtype=x.dtype); return torch.linalg.cholesky(K)
    def nll(self,x,y):
        L=self.factor(x); alpha=torch.cholesky_solve(y[:,None],L)
        return .5*(y[:,None]*alpha).sum()+L.diag().log().sum()+.001*sum(p.square().sum() for p in self.encoder.parameters())

def fit_predict(X,y,pool,seed):
    torch.manual_seed(int(seed)); model=DKL(); opt=torch.optim.Adam(model.parameters(),lr=.015)
    tensor=lambda x: torch.as_tensor(x,dtype=torch.double)
    x=tensor(np.asarray(X)); z=tensor(np.asarray(pool))
    center=float(np.mean(y)); scale=max(float(np.std(y)),1e-5); yy=tensor((np.asarray(y)-center)/scale)
    for _ in range(40):
        opt.zero_grad(); loss=model.nll(x,yy); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),10.); opt.step()
        with torch.no_grad():
            model.log_noise.clamp_(-8,1); model.log_length.clamp_(-3,3); model.log_amp.clamp_(-4,4)
    with torch.no_grad():
        L=model.factor(x); k=model.kernel(x,z); alpha=torch.cholesky_solve(yy[:,None],L); mu=(k.T@alpha).squeeze(1)
        v=torch.linalg.solve_triangular(L,k,upper=False); var=(model.log_amp.exp()-v.square().sum(0)).clamp_min(1e-9)
    return mu.numpy()*scale+center,var.sqrt().numpy()*scale


def nd(a):
    a=np.asarray(a);return np.array([not np.any(np.all(a<=v,axis=1)&np.any(a<v,axis=1)) for v in a],bool)
def hv(a,ref):
    a=np.asarray(a,float).reshape(-1,2);a=a[np.all(np.isfinite(a),axis=1)&np.all(a<ref,axis=1)]
    if not len(a):return 0.
    a=a[nd(a)];a=a[np.argsort(a[:,0])];prev=ref[1];area=0.
    for x,y in a:
        if y<prev:area+=(ref[0]-x)*(prev-y);prev=y
    return area/np.prod(ref)
def ref_for(metric):return np.array([REF[0],EXPOSURE_REF if metric=='exposure_mAh_min' else REF[1]])
def points(rows,metric):
    return np.array([[r['time_min'],r[metric]] for r in rows if r['feasible']],float).reshape(-1,2)
def ranks(rows):
    ranks=np.full(len(rows),np.inf); crowd=np.zeros(len(rows)); ids=np.flatnonzero([r['feasible'] for r in rows]); a=np.array([[rows[i]['time_min'],rows[i][OBJECTIVE]] for i in ids]);rank=0
    while len(ids):
        mask=nd(a);ix=ids[mask];v=a[mask];ranks[ix]=rank
        for j in range(2):
            order=np.argsort(v[:,j]);crowd[ix[order[[0,-1]]]]=np.inf
            span=np.ptp(v[:,j])
            if len(ix)>2 and span>0:crowd[ix[order[1:-1]]]+=(v[order[2:],j]-v[order[:-2],j])/span
        ids,a=ids[~mask],a[~mask];rank+=1
    for i,r in enumerate(rows):
        if not r['feasible']:ranks[i]=rank+r['violation']
    return ranks,crowd
def candidate(rows,rng,f,rc=None):
    if rng.random()<.15:return random_protocol(rng,f),'random'
    rank,crowd=rc if rc is not None else ranks(rows)
    def parent():
        ix=rng.choice(len(rows),min(3,len(rows)),replace=False);i=min(ix,key=lambda i:(rank[i],-crowd[i]));return rows[i]['protocol']
    p=parent();op='copy'
    if rng.random()<.7:p,op=cross(p,parent(),rng)
    if rng.random()<.85:p,op2=mutate(p,rng);op+='+'+op2
    return p,op
def acquire(rows,pool,rng,seed):
    ids=[i for i,r in enumerate(rows) if r['feasible']]
    if len(ids)<5:return 0
    X=np.array([encode(r['protocol']) for r in rows]); Z=np.array([encode(p) for p,_ in pool])
    a=points(rows,OBJECTIVE); ref=ref_for(OBJECTIVE);base=hv(a,ref)
    # log1p targets keep nonnegative predictions and handle orders of magnitude.
    pred=[fit_predict(X[ids],np.log1p(a[:,j]),Z,seed+j) for j in range(2)]
    if len(ids)==len(rows):pf=np.ones(len(pool))
    else:
        mu,sd=fit_predict(X,np.array([float(r['feasible']) for r in rows]),Z,seed+2)
        pf=norm.cdf((mu-.5)/np.maximum(sd,.05))
    score=[]
    for i in range(len(pool)):
        draws=np.column_stack([np.maximum(0,np.expm1(np.clip(rng.normal(mu[i],sd[i],48),-20,20))) for mu,sd in pred])
        score.append(pf[i]*np.mean([max(0,hv(np.vstack([a,q]),ref)-base) for q in draws]))
    return int(np.argmax(score))
assert abs(hv([[1,3],[2,2]],np.array([4.,4.]))-5/16)<1e-12


## Live generation outputs
Each panel shows all feasible evaluated points, current-generation points, and cumulative observed nondominated front. A second figure shows representative primary-front protocols and their actual electrochemical responses, with control-mode shading. Generation summaries and all front memberships are exported immediately. Trace figures use stored simulation traces, not additional model calls.

In [ ]:
def front_plot(rows,title,folder,g):
    fig,ax=plt.subplots(2,2,figsize=(12,8))
    for p,(metric,label) in zip(ax.flat,RISK.items()):
        a=points(rows,metric)
        if len(a):
            p.scatter(*a.T,s=15,alpha=.3,label='Archive')
            b=points([r for r in rows if r['generation']==g],metric)
            if len(b):p.scatter(*b.T,s=24,marker='x',label='New')
            f=a[nd(a)];f=f[np.argsort(f[:,0])];p.step(*f.T,where='post',label='Observed front');p.scatter(*f.T,s=22)
        p.set(xlabel='Time to target (min)',ylabel=label);p.grid(alpha=.2)
        if len(a):p.legend(fontsize=8)
    fig.suptitle(title);fig.tight_layout();fig.savefig(folder/f'front_{g:03d}.png',dpi=140);plt.show();plt.close(fig)
def trace_plot(rows,folder,g):
    ok=[r for r in rows if r['feasible']]
    if not ok:return
    a=points(ok,OBJECTIVE);ids=np.flatnonzero(nd(a));ids=ids[np.argsort(a[ids,0])];ids=ids[np.unique(np.linspace(0,len(ids)-1,min(3,len(ids))).astype(int))]
    fig,ax=plt.subplots(3,len(ids),figsize=(5*len(ids),8),squeeze=False)
    for col,i in enumerate(ids):
        r=ok[i];d=r['trace'];t=d['t']/60
        ax[0,col].plot(t,d['V']);ax[0,col].set(ylabel='Voltage (V)',title=f"{r['time_min']:.1f} min | {r[OBJECTIVE]:.3g}")
        ax[1,col].plot(t,-d['I']);ax[1,col].set(ylabel='Charge current (A)')
        ax[2,col].plot(t,d['Q'],label='Metal');ax[2,col].plot(t,d['D'],label='Dead');ax[2,col].set(ylabel='Inventory (mAh)',xlabel='Time (min)');ax[2,col].legend()
        for mode in ['I','V']:
            mask=d['mode']==mode
            for p in ax[:,col]:p.fill_between(t,0,1,where=mask,transform=p.get_xaxis_transform(),alpha=.07,color='blue' if mode=='I' else 'green')
    fig.tight_layout();fig.savefig(folder/f'traces_{g:03d}.png',dpi=140);plt.show();plt.close(fig)
def snapshot(rows,folder,g):
    frame=pd.DataFrame([{k:(json.dumps(v) if k=='protocol' else v) for k,v in r.items() if k!='trace'} for r in rows])
    frame.to_csv(folder/'evaluations.csv',index=False)
    summary=dict(generation=g,calls=len(rows),feasible_fraction=float(frame.feasible.mean()),failures=int((frame.error!='').sum()))
    for metric in RISK:
        f=frame[frame.feasible].copy();a=f[['time_min',metric]].to_numpy()
        f=f.loc[nd(a)] if len(a) else f
        f.sort_values('time_min').to_csv(folder/f'{metric}_front_{g:03d}.csv',index=False)
        summary['hv_'+metric]=hv(a,ref_for(metric))
    return summary
def campaign(method,f,seed,shared):
    rng=np.random.default_rng(seed+1000);folder=OUT/f/f'{method}_{seed}';folder.mkdir(parents=True,exist_ok=True)
    rows=[dict(r,generation=0,call=i+1,operation='initial') for i,r in enumerate(shared)];hist=[]
    for g in range(N_GEN+1):
        if g:
            for _ in range(OFFSPRING):
                if method=='Random':p,op=random_protocol(rng,f),'random'
                else:
                    rc=ranks(rows)
                    if method=='GA':p,op=candidate(rows,rng,f,rc)
                    else:
                        pool=[candidate(rows,rng,f,rc) for _ in range(POOL)]
                        try:idx=acquire(rows,pool,rng,seed+len(rows))
                        except (RuntimeError,ValueError,np.linalg.LinAlgError) as e:print('DKL fallback:',e);idx=0
                        p,op=pool[idx]
                r=evaluate(p,trace=True);rows.append(dict(r,protocol=p,generation=g,call=len(rows)+1,operation=op))
        s=snapshot(rows,folder,g);hist.append(s);pd.DataFrame(hist).to_csv(folder/'history.csv',index=False)
        print(f'{f} | {method} | seed {seed} | generation {g}:',s,flush=True)
        front_plot(rows,f'{f} / {method} / seed {seed} / generation {g}',folder,g);trace_plot(rows,folder,g)
    return hist,rows


## Execute campaigns
Default: 3 representations × 3 methods × 32 logical simulator calls = 288 logical protocol evaluations per run. Shared initial designs are physically executed once per representation and replicate. Each protocol may require several segment solves. Generation plots remain in the notebook output and are saved to disk.

In [ ]:
histories=[];all_rows=[]
for run in range(N_RUNS):
    seed=BASE_SEED+run
    for f in FAMILIES:
        shared=[dict(evaluate(p,trace=True),protocol=p) for p in initial(np.random.default_rng(seed),f)]
        for method in METHODS:
            h,rows=campaign(method,f,seed,shared)
            histories.extend([dict(x,method=method,family=f,seed=seed) for x in h])
            all_rows.extend([dict(x,method=method,family=f,seed=seed) for x in rows])
            pd.DataFrame(histories).to_csv(OUT/'all_histories.csv',index=False)


## Final comparison and ensemble convergence
Hypervolume is normalized by the fixed reference rectangle; points outside it are not credited, but remain in plotted fronts. Inspect sensitivity to the chosen reference. A single run has individual curves; multiple runs show median and interquartile bands. Pooled final points describe an empirical archive, not the global optimum.

In [ ]:
h=pd.DataFrame(histories)
fig,ax=plt.subplots(2,2,figsize=(13,9))
for p,metric in zip(ax.flat,RISK):
    for (f,method),q in h.groupby(['family','method']):
        a=q.pivot(index='seed',columns='calls',values='hv_'+metric)
        x=a.columns.to_numpy();line=p.plot(x,a.median().to_numpy(),label=f'{f}/{method}')[0]
        if len(a)>1:p.fill_between(x,a.quantile(.25).to_numpy(),a.quantile(.75).to_numpy(),alpha=.15,color=line.get_color())
    p.set(xlabel='Protocol evaluations',ylabel='Normalized HV',title=RISK[metric]);p.legend(fontsize=6);p.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'hypervolume_comparison.png',dpi=180);plt.show()
fig,ax=plt.subplots(2,2,figsize=(13,9))
for p,metric in zip(ax.flat,RISK):
    for f in FAMILIES:
        for method in METHODS:
            a=points([r for r in all_rows if r['family']==f and r['method']==method],metric)
            if len(a):
                a=a[nd(a)];a=a[np.argsort(a[:,0])];p.plot(*a.T,'o-',ms=3,label=f'{f}/{method}')
    p.set(xlabel='Time (min)',ylabel=RISK[metric]);p.legend(fontsize=6);p.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'final_fronts.png',dpi=180);plt.show()
import shutil
shutil.make_archive(str(OUT),'zip',OUT)
print('Results:',str(OUT)+'.zip')


## Limitations and next steps
Risk metrics are nominal continuum-model proxies. Gross deposition is positive part of net local flux, not separate microscopic forward reaction rate. Check sign and mass balance residual, spatial/temporal convergence, switching spikes, and feasibility thresholds. Voltage-only and scheduled CC–CV seeds are not a fully optimized event-driven CC–CV baseline. No response-preserving mode flip is implemented in this revision. Fourier projection during local mutations can smooth spikes. Fixed horizon, charging-only current commands, normalized repair, fixed Fourier order and fixed hypervolume reference are benchmark choices. For independently optimized secondary fronts set OBJECTIVE to peak_mAh, dead_mAh or exposure_mAh_min and use a separate output directory. Compare uncertainty and robustness before physical use.